# 📄 Module 06: Document Loaders & Text Splitters

---

## What Are Document Loaders?

**Document Loaders** import data from various sources into LangChain's `Document` format, ready for processing.

```
PDF / Word / CSV / URL / Database / S3 / YouTube
                   ↓  Document Loader
         Document(page_content=..., metadata={...})
                   ↓  Text Splitter
      [Chunk 1, Chunk 2, Chunk 3, ...]  ← Ready for embedding
```

---

## Available Document Loaders

| Loader | Source | Package |
|--------|--------|----------|
| `PyPDFLoader` | PDF files | `pypdf` |
| `Docx2txtLoader` | Word documents | `docx2txt` |
| `CSVLoader` | CSV files | built-in |
| `WebBaseLoader` | Web pages | `beautifulsoup4` |
| `WikipediaLoader` | Wikipedia | `wikipedia` |
| `YoutubeLoader` | YouTube transcripts | `youtube-transcript-api` |
| `UnstructuredLoader` | Many formats | `unstructured` |
| `DirectoryLoader` | Entire folders | built-in |
| `JSONLoader` | JSON files | `jq` |
| `TextLoader` | Plain text | built-in |
| `GitLoader` | Git repositories | `gitpython` |

---

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

# Install loaders
%pip install -q pypdf docx2txt beautifulsoup4 requests
print("Packages installed ✅")

## 1️⃣ Document Structure

In [ ]:
from langchain_core.documents import Document

# ============================================================
# The Document object — the core data unit in LangChain RAG
# ============================================================
doc = Document(
    page_content="LangChain is an open-source framework for building LLM-powered applications. It provides abstractions for working with models, prompts, chains, agents, and memory.",
    metadata={
        "source": "langchain_docs.pdf",
        "page": 1,
        "author": "Harrison Chase",
        "date": "2024-01-01",
        "category": "framework"
    }
)

print("page_content:", doc.page_content)
print("\nmetadata:", doc.metadata)
print("\nContent length:", len(doc.page_content), "characters")

## 2️⃣ TextLoader — Load Plain Text Files

In [ ]:
from langchain_community.document_loaders import TextLoader
import tempfile
import os

# Create a sample text file
sample_text = """
Introduction to Machine Learning
================================

Machine learning is a subset of artificial intelligence that enables computers
to learn from data without being explicitly programmed.

Types of Machine Learning:
1. Supervised Learning - Learning from labeled data
2. Unsupervised Learning - Finding patterns in unlabeled data
3. Reinforcement Learning - Learning through rewards and penalties

Common Algorithms:
- Linear Regression
- Decision Trees
- Neural Networks
- Support Vector Machines
"""

# Write to temp file
with open("sample_ml.txt", "w") as f:
    f.write(sample_text)

# Load it!
loader = TextLoader("sample_ml.txt", encoding="utf-8")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Content length: {len(docs[0].page_content)} chars")
print(f"Metadata: {docs[0].metadata}")
print(f"\nContent preview:\n{docs[0].page_content[:200]}...")

## 3️⃣ CSVLoader — Load Tabular Data

In [ ]:
from langchain_community.document_loaders import CSVLoader
import csv

# Create a sample CSV
csv_data = [
    ["name", "role", "experience_years", "skills"],
    ["Alice Smith", "ML Engineer", "5", "Python, TensorFlow, PyTorch"],
    ["Bob Johnson", "Data Scientist", "3", "Python, R, SQL, Spark"],
    ["Carol Williams", "AI Researcher", "8", "Deep Learning, NLP, Computer Vision"],
    ["David Brown", "Backend Engineer", "4", "Go, Kubernetes, PostgreSQL"]
]

with open("team.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

# Load CSV — each row becomes a Document!
loader = CSVLoader(
    file_path="team.csv",
    content_columns=["name", "role", "skills"],  # Columns to include in content
    metadata_columns=["experience_years"]         # Columns to use as metadata
)

docs = loader.load()

print(f"Loaded {len(docs)} rows as documents")
print()
for doc in docs:
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

## 4️⃣ WebBaseLoader — Load Web Pages

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# ============================================================
# Load content from any web URL
# ============================================================
loader = WebBaseLoader(
    web_paths=["https://en.wikipedia.org/wiki/Large_language_model"],
    # bs_kwargs={"parse_only": SoupStrainer(class_="mw-body-content")}  # Optional CSS filter
)

docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Source: {docs[0].metadata.get('source', 'N/A')}")
print(f"Content length: {len(docs[0].page_content)} chars")
print(f"\nContent preview (first 500 chars):\n")
print(docs[0].page_content[:500])

In [ ]:
# ============================================================
# Load multiple URLs at once
# ============================================================
loader = WebBaseLoader([
    "https://python.langchain.com/docs/introduction/",
    "https://python.langchain.com/docs/concepts/"
])

all_docs = loader.load()
print(f"Loaded {len(all_docs)} pages")
for doc in all_docs:
    print(f"  - {doc.metadata.get('source', 'N/A')}: {len(doc.page_content)} chars")

## 5️⃣ PyPDFLoader — Load PDF Files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# ============================================================
# Load a PDF (each page becomes a Document)
# ============================================================
# Replace with your PDF path
# loader = PyPDFLoader("path/to/your/document.pdf")

# Demo with a public PDF
loader = PyPDFLoader("https://arxiv.org/pdf/1706.03762")
docs = loader.load()

print(f"Loaded {len(docs)} pages from PDF")
print(f"\nPage 1 metadata: {docs[0].metadata}")
print(f"\nPage 1 content (first 300 chars):")
print(docs[0].page_content[:300])

## 6️⃣ DirectoryLoader — Load Entire Folders

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# Create sample directory with multiple files
import os
os.makedirs("docs_sample", exist_ok=True)

files = {
    "docs_sample/python.txt": "Python is a high-level, interpreted programming language known for its simplicity.",
    "docs_sample/javascript.txt": "JavaScript is a scripting language used primarily for web development.",
    "docs_sample/rust.txt": "Rust is a systems programming language focused on safety, speed, and concurrency."
}

for path, content in files.items():
    with open(path, "w") as f:
        f.write(content)

# Load all .txt files from directory
loader = DirectoryLoader(
    path="docs_sample",
    glob="**/*.txt",          # Pattern to match
    show_progress=True,       # Progress bar
    use_multithreading=True   # Parallel loading
)

docs = loader.load()
print(f"\nLoaded {len(docs)} documents from directory")
for doc in docs:
    print(f"  - {doc.metadata['source']}: '{doc.page_content[:50]}...'")

## 7️⃣ Text Splitters — Chunk Your Documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# RecursiveCharacterTextSplitter — The most important splitter!
# It tries to split at these separators in order:
# 1. \n\n (paragraphs)
# 2. \n (lines)
# 3. . (sentences)
# 4. " " (words)
# 5. "" (characters)
# ============================================================

long_text = """
Machine learning is a subset of artificial intelligence (AI) that focuses on building systems that can learn from data.

Types of Machine Learning:

Supervised Learning involves training a model on labeled data. The model learns to map inputs to outputs by studying examples. Common applications include email spam detection, image classification, and credit scoring.

Unsupervised Learning finds hidden patterns in data without labels. Clustering algorithms group similar data points together. Dimensionality reduction techniques compress data while preserving structure. Examples include customer segmentation and anomaly detection.

Reinforcement Learning trains agents to make decisions through trial and error. The agent receives rewards for good actions and penalties for bad ones. This approach powers game-playing AI like AlphaGo and robotics applications.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       # Maximum characters per chunk
    chunk_overlap=50,     # Overlap between consecutive chunks (for context)
    length_function=len,  # How to measure chunk size
    separators=["\n\n", "\n", ". ", " ", ""]  # Try these in order
)

chunks = splitter.split_text(long_text)

print(f"Original text: {len(long_text)} chars")
print(f"Split into {len(chunks)} chunks")
print()
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars):")
    print(f"  '{chunk[:100]}...'")
    print()

In [ ]:
# ============================================================
# Split Documents (not just text)
# ============================================================
docs_to_split = loader.load()  # From directory loader above

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

# split_documents preserves metadata!
split_docs = splitter.split_documents(docs_to_split)

print(f"Original: {len(docs_to_split)} documents")
print(f"After splitting: {len(split_docs)} chunks")
print()
for doc in split_docs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}")
    print()

In [ ]:
# ============================================================
# TokenTextSplitter — Split by tokens (more accurate for LLMs)
# ============================================================
# %pip install -q tiktoken

from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(
    chunk_size=100,     # Max 100 TOKENS per chunk
    chunk_overlap=10,   # 10 token overlap
    model_name="gpt-4"  # Use GPT-4's tokenizer
)

token_chunks = token_splitter.split_text(long_text)
print(f"Token-based splitting: {len(token_chunks)} chunks")
for i, chunk in enumerate(token_chunks):
    print(f"Chunk {i+1}: {chunk[:80]}...")

In [ ]:
# ============================================================
# Markdown Splitter — Splits at headers
# ============================================================
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """
# Introduction to Python

Python is a versatile programming language.

## Variables and Data Types

Python supports integers, floats, strings, and booleans.
Variables don't need explicit type declarations.

## Control Flow

Python uses if/elif/else for conditions and for/while for loops.
Indentation defines code blocks.

### For Loops

For loops iterate over sequences like lists and ranges.

### While Loops

While loops repeat while a condition is True.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

md_docs = md_splitter.split_text(markdown_doc)

print(f"Split into {len(md_docs)} sections:")
for doc in md_docs:
    print(f"\nMetadata: {doc.metadata}")
    print(f"Content: {doc.page_content[:100]}")

## 8️⃣ Chunking Strategy — Choosing the Right Parameters

```
┌─────────────────────────────────────────────────────────────────┐
│              CHUNKING STRATEGY GUIDE                            │
├─────────────────────────────────────────────────────────────────┤
│ chunk_size = 200-500  → Short, precise retrieval               │
│ chunk_size = 500-1500 → Balanced (recommended for most cases)   │
│ chunk_size = 1500+    → Long, more context per chunk            │
├─────────────────────────────────────────────────────────────────┤
│ chunk_overlap = 10-15% of chunk_size (typical recommendation)  │
│ chunk_overlap = 50-200 chars for text                          │
│ chunk_overlap = 20-50 tokens for token-based                   │
├─────────────────────────────────────────────────────────────────┤
│ Use Case → Recommended Splitter                                 │
│ General text    → RecursiveCharacterTextSplitter               │
│ Markdown/Docs   → MarkdownHeaderTextSplitter                   │
│ Code            → RecursiveCharacterTextSplitter (code mode)   │
│ Token budget    → TokenTextSplitter                            │
└─────────────────────────────────────────────────────────────────┘
```

## ✅ Module 06 Summary

You've learned:
- ✅ The `Document` data structure
- ✅ `TextLoader`, `CSVLoader`, `WebBaseLoader`, `PyPDFLoader`, `DirectoryLoader`
- ✅ `RecursiveCharacterTextSplitter` (the gold standard)
- ✅ `TokenTextSplitter` for LLM token budgets
- ✅ `MarkdownHeaderTextSplitter` for structured documents
- ✅ Chunking strategy guidelines

### 🚀 Next: [Module 07 — Embeddings & Vector Stores](07_Embeddings_and_VectorStores.ipynb)

In [ ]:
# Cleanup temp files
import os, shutil
for f in ["sample_ml.txt", "team.csv"]:
    if os.path.exists(f): os.remove(f)
if os.path.exists("docs_sample"): shutil.rmtree("docs_sample")
print("Cleaned up temp files ✅")